# Phân tích cảm xúc tiếng Việt trên VLSP

So sánh TF-IDF + LinearSVC, CBOW + GRU và PhoBERT-base-v2 trên Base và Augmented với cùng tập test. Nhãn gốc là −1 (tiêu cực), 0 (trung lập) và +1 (tích cực). Notebook kiểm tra dữ liệu, phân tích văn bản gốc và sau chuẩn hóa, rồi xuất chỉ số và ma trận nhầm lẫn từ dự đoán thật.

- **Dữ liệu:** [VLSP Sentiment Dataset trên Kaggle](https://www.kaggle.com/datasets/anhphmduy/vlsp-sentiment-dataset). Có thể đặt các CSV và `vi-model-CBOW.bin` trong `./data` hoặc chỉ đường dẫn ở ô cấu hình.
- **Tăng cường:** Nếu có bảng nguồn gốc từng dòng Augmented, dùng cột `aug_row,base_row` (chỉ số dòng dữ liệu bắt đầu từ 0, bỏ qua header). Notebook dùng cùng một validation gốc cho cả hai phiên bản và giữ các câu phái sinh của validation ngoài tập fit. Nếu thiếu bảng nguồn gốc, notebook dùng số epoch và siêu tham số cố định trên toàn bộ dữ liệu phát triển, không chọn theo validation hay test.
- **Môi trường:** Google Colab với GPU cho hai nhánh neural, có Internet ở lần tải đầu. PhoBERT dùng RDRSegmenter của VnCoreNLP, cần Java. Chạy lại từ đầu khi đổi dữ liệu hoặc cấu hình.

Đặt `RUN_TRAINING = False` ở ô chọn chế độ đầu notebook rồi Run all để xuất EDA. Chế độ này không tải vector CBOW hoặc trọng số mô hình phân loại. Hình độ dài BPE vẫn cần Java, VnCoreNLP và tokenizer PhoBERT. Notebook ghi hình và bảng EDA vào thư mục riêng trong `vlsp_results`.

Tham khảo [BBC News EDA](https://ltsach.github.io/AILearningHub/01_Data_Analysis/01_EDA/bbcnews_text_classification/eda_report_tutorial.html) cho cách tách thống kê văn bản gốc khỏi phân tích từ vựng; xem [pipeline TF-IDF](https://ltsach.github.io/AILearningHub/04_Natural_Language_Processing/text_classification/pages/traditional_ml/index.html) và [BERT](https://ltsach.github.io/AILearningHub/04_Natural_Language_Processing/text_classification/pages/bert/index.html) cho cách trình bày từng khâu.

In [ ]:
RUN_TRAINING = True  # False: chỉ xuất EDA, không huấn luyện.

In [ ]:
%pip install -q "kagglehub>=0.3,<2" "pyvi==0.1.1" "scikit-learn>=1.4,<2" "pandas>=2,<4" "matplotlib>=3.8,<4" "transformers>=4.52,<5" "py-vncorenlp==0.1.4"
if RUN_TRAINING:
    %pip install -q "torch>=2.6,<3" "gensim>=4.4,<5" "datasets>=3,<5" "accelerate>=1,<2"

In [ ]:
import gc
import hashlib
import json
import os
import random
import re
import shutil
import time
import unicodedata
from collections import Counter
from itertools import groupby
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score, matthews_corrcoef,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.svm import LinearSVC

SEED = 42
LABELS = [-1, 0, 1]
DATA_DIR = None  # e.g. Path("/content/data")
AUGMENTATION_LINEAGE = None  # e.g. Path("/content/data/augmentation_lineage.csv")
WORD2VEC_FILE = None  # e.g. Path("/content/vi-model-CBOW.bin")
DATASET_HANDLE = "anhphmduy/vlsp-sentiment-dataset/versions/5"
OUTPUT_PARENT = Path("vlsp_results")
TFIDF_SVD_COMPONENTS = 300
W2V_MAX_LENGTH = 256
GRU_HIDDEN_SIZE = 64
GRU_BATCH_SIZE = 64
GRU_EPOCHS = 4
PHOBERT_MAX_LENGTH = 256
PHOBERT_MODEL = "vinai/phobert-base-v2"

random.seed(SEED)
np.random.seed(SEED)

## 1. Nạp dữ liệu và kiểm tra schema

Notebook nhận hai tên file Augmented và kiểm tra cả tab lẫn dấu phẩy theo header. Dòng dữ liệu nhận ID theo tên tập và thứ tự trong file gốc. File thiếu, schema sai và nhãn ngoài tập quy định được báo lỗi. Kích thước hiển thị lấy trực tiếp từ dữ liệu đã đọc.

In [ ]:
FILES = {
    "Base": ("vlsp_sentiment_train.csv",),
    "Augmented": ("vlsp_sentiment_train_augmented_v2.csv", "augmented_v2.csv"),
    "Test": ("vlsp_sentiment_test.csv",),
}

def locate_inputs(root):
    return {
        name: next((root / filename for filename in filenames if (root / filename).is_file()), None)
        for name, filenames in FILES.items()
    }

def resolve_inputs():
    if DATA_DIR is not None:
        root = Path(DATA_DIR).resolve()
    else:
        candidates = [
            Path("data"), Path("."),
            Path("/kaggle/input/vlsp-sentiment-dataset"),
            Path("/kaggle/input/datasets/anhphmduy/vlsp-sentiment-dataset"),
        ]
        root = next((p for p in candidates if all(locate_inputs(p).values())), None)
        if root is None:
            import kagglehub
            root = Path(kagglehub.dataset_download(DATASET_HANDLE))
            root = next(
                (p.parent for p in sorted(root.rglob(FILES["Base"][0]))
                 if all(locate_inputs(p.parent).values())),
                root,
            )
        root = root.resolve()
    paths = locate_inputs(root)
    missing = ["/".join(FILES[name]) for name, path in paths.items() if path is None]
    if missing:
        raise FileNotFoundError(f"Thiếu dữ liệu VLSP tại {root}: {', '.join(missing)}")
    return root, paths

def read_vlsp(path, split):
    for delimiter in ("\t", ","):
        header = pd.read_csv(path, sep=delimiter, encoding="utf-8-sig", nrows=0)
        if {"Class", "Data"}.issubset(header.columns.str.strip()):
            break
    else:
        raise ValueError(f"{path.name} cần hai cột Class và Data, phân tách bằng tab hoặc dấu phẩy.")
    frame = pd.read_csv(path, sep=delimiter, encoding="utf-8-sig", keep_default_na=False, dtype=str)
    frame.columns = frame.columns.str.strip()
    frame = frame[["Class", "Data"]].copy()
    labels = pd.to_numeric(frame["Class"], errors="coerce")
    invalid = ~labels.isin(LABELS)
    if frame.empty or invalid.any():
        raise ValueError(f"{path.name}: {int(invalid.sum())} nhãn sai hoặc tập rỗng.")
    frame["label"] = labels.astype(int)
    frame = frame.rename(columns={"Data": "text"})
    frame["row"] = np.arange(len(frame))
    frame["sample_id"] = split.lower() + "-" + frame["row"].astype(str)
    return frame[["sample_id", "row", "label", "text"]]

data_root, paths = resolve_inputs()
raw = {name: read_vlsp(path, name) for name, path in paths.items()}
display(pd.DataFrame({name: frame["label"].value_counts() for name, frame in raw.items()})
        .reindex(LABELS).fillna(0).astype(int))

## 2. Độ dài văn bản gốc

Hai đường phân phối tích lũy theo nhãn cho biết bao nhiêu phần trăm nhận xét Base gốc có độ dài không quá một giá trị trên trục ngang. Đọc đường ở 50% và 95% để hiểu độ dài điển hình và phần đuôi. Ngưỡng 256 được kiểm tra bằng tokenizer thực, vì đơn vị cách trắng và ký tự chưa phải số token.

In [ ]:
import tempfile

OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)
eda_dir = Path(tempfile.mkdtemp(prefix="eda-", dir=OUTPUT_PARENT))

def save_text_figure(figure, filename):
    figure.savefig(eda_dir / filename, dpi=140, bbox_inches="tight")
    plt.show()
    plt.close(figure)

base_raw = raw["Base"]
lengths = pd.DataFrame({
    "label": base_raw["label"],
    "characters": base_raw["text"].str.len(),
    "whitespace_units": base_raw["text"].str.split().str.len(),
})
raw_length_summary = lengths.groupby("label")[["characters", "whitespace_units"]].quantile([0.5, 0.95, 1])
raw_length_summary = raw_length_summary.rename_axis(["label", "quantile"])
raw_length_summary.to_csv(eda_dir / "raw_length_quantiles.csv")
display(raw_length_summary.round(1))

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for ax, field, unit in zip(axes, ("characters", "whitespace_units"), ("Ký tự", "Đơn vị cách trắng")):
    for label, frame in lengths.groupby("label"):
        values = np.sort(frame[field].to_numpy())
        ax.step(1 + values, 100 * np.arange(1, len(values) + 1) / len(values), where="post", label=str(label))
    ax.set(xscale="log", xlabel=f"{unit} + 1 (thang log)", ylabel="Tỷ lệ tích lũy (%)", title=f"Base gốc · {unit}")
    ax.legend(title="Nhãn")
save_text_figure(fig, "eda-raw-lengths.png")

## 3. Chuẩn hóa và kiểm toán dữ liệu

Giữ dấu tiếng Việt, chữ số và từ phủ định. Chuẩn Unicode NFC, đổi các emoji và emoticon có trong từ điển, giảm ký tự lặp từ ba lần trở lên và chuẩn hóa teencode theo ranh giới từ. Một số cách viết phụ thuộc ngữ cảnh nên từ điển chỉ giữ các ánh xạ tương đối ít mơ hồ.

Giữ test cố định. Dòng test rỗng sau chuẩn hóa dừng lượt chạy để xử lý dữ liệu nguồn. Loại khỏi tập phát triển các dòng rỗng, dòng có văn bản trùng test và nhóm văn bản chuẩn hóa có nhãn mâu thuẫn trong dữ liệu phát triển. Sau đó giữ một bản cho mỗi văn bản chuẩn hóa trong từng phiên bản Base/Augmented. Hash trùng văn bản chỉ kiểm tra khớp chính xác sau các phép chuẩn hóa.

In [ ]:
TEENCODE = {
    "ko": "không", "hok": "không", "khg": "không", "đc": "được", "dc": "được",
    "sp": "sản phẩm", "đt": "điện thoại", "thik": "thích", "iu": "yêu",
    "mik": "mình", "vs": "với", "bthg": "bình thường", "cx": "cũng",
}
EMOTICONS = {
    "❤️": "yêu thích", "👍": "tốt", "😊": "vui vẻ", "😍": "yêu thích",
    "👎": "chê", "😡": "tức giận", "😭": "khóc", "💔": "thất vọng",
    ":)": "cười", ":-)": "cười", ":(": "buồn", ":-(": "buồn",
    ":d": "cười", "^^": "vui", "t.t": "khóc",
}
EMOTICON_RE = re.compile("|".join(map(re.escape, sorted(EMOTICONS, key=len, reverse=True))))
TEENCODE_RE = re.compile(r"\b(?:" + "|".join(map(re.escape, TEENCODE)) + r")\b")

VIETNAMESE_TONES = dict.fromkeys(map(ord, "\u0300\u0301\u0303\u0309\u0323"))

def without_tone(character):
    return unicodedata.normalize("NFC", unicodedata.normalize("NFD", character).translate(VIETNAMESE_TONES))

def collapse_elongation(text):
    output = []
    for base, grouped in groupby(text, key=without_tone):
        run = "".join(grouped)
        tones = {character for character in run if character != base}
        if base.isalpha() and len(run) >= 3 and len(tones) <= 1:
            output.append(next(iter(tones), base))
        else:
            output.append(run)
    return "".join(output)

def normalize_text(text):
    text = unicodedata.normalize("NFC", text).lower()
    text = EMOTICON_RE.sub(lambda match: " " + EMOTICONS[match[0]] + " ", text)
    text = collapse_elongation(text)
    text = TEENCODE_RE.sub(lambda match: TEENCODE[match[0]], text)
    text = re.sub(r"[^\w\s.,!?]", " ", text)
    return " ".join(text.split())

assert normalize_text("Ko, ngonnnnn :D ❤️") == "không, ngon cười yêu thích"
assert normalize_text("Điện thoại 5G, quáaaa!") == "điện thoại 5g, quá!"

def text_key(text):
    return " ".join(re.findall(r"\w+", text.replace("_", " ").casefold()))

def audit_inputs(frames):
    prepared = {
        name: frame.assign(
            text_clean=frame["text"].map(normalize_text),
            raw_key=frame["text"].map(
                lambda text: " ".join(unicodedata.normalize("NFC", text).casefold().split())
            ),
        )
        for name, frame in frames.items()
    }
    for frame in prepared.values():
        frame["clean_key"] = frame["text_clean"].map(text_key)

    test_frame = prepared["Test"]
    if test_frame["clean_key"].eq("").any():
        raise ValueError("Test có văn bản rỗng sau chuẩn hóa. Kiểm tra các dòng nguồn trước khi chạy.")

    audit = []
    development = pd.concat([prepared["Base"], prepared["Augmented"]])
    conflicting = set(development.groupby("clean_key")["label"].nunique().loc[lambda s: s > 1].index)
    cleaned = {"Test": test_frame.copy()}
    for name, frame in prepared.items():
        empty = frame["clean_key"].eq("")
        raw_overlap = frame["raw_key"].isin(test_frame["raw_key"]) & ~empty
        overlap = frame["clean_key"].isin(test_frame["clean_key"]) & ~empty
        conflict = frame["clean_key"].isin(conflicting) & ~empty & ~raw_overlap & ~overlap
        entry = {
            "split": name,
            "input_rows": len(frame),
            "raw_blank": int(frame["raw_key"].eq("").sum()),
            "empty_after_cleaning": int(empty.sum()),
            "raw_duplicate_extras": int(frame.loc[~empty, "raw_key"].duplicated().sum()),
            "normalized_duplicate_extras": int(frame.loc[~empty, "clean_key"].duplicated().sum()),
            "raw_overlap_with_test": None if name == "Test" else int(raw_overlap.sum()),
            "normalized_overlap_with_test": None if name == "Test" else int(overlap.sum()),
            "conflicting_dev_labels": None if name == "Test" else int(conflict.sum()),
        }
        if name != "Test":
            cleaned[name] = frame.loc[~(empty | raw_overlap | overlap | conflict)].copy()
        else:
            entry["test_conflicting_keys"] = int((frame.groupby("clean_key")["label"].nunique() > 1).sum())
        audit.append(entry)
    return cleaned, pd.DataFrame(audit).set_index("split")

cleaned, audit = audit_inputs(raw)
display(audit)
examples = cleaned["Base"].loc[lambda f: f["text"].ne(f["text_clean"]), ["text", "text_clean"]]
display(examples.head(5))

## 4. Tách tập và kiểm tra dữ liệu sau lọc

Khi có lineage, mỗi dòng Augmented phải chỉ tới một dòng Base hợp lệ với cùng nhãn. Notebook dành một fold stratified cho validation theo nhóm văn bản Base chuẩn hóa, loại khỏi Augmented mọi dòng có cùng nhóm nguồn hoặc cùng văn bản chuẩn hóa với validation. Chỉ sau bước đó mới gộp các dòng trùng trong mỗi tập fit. Các thí nghiệm dùng chung validation và test. Khi lineage chưa có, dùng lịch huấn luyện cố định cho cả hai phiên bản và ghi rõ chế độ trong kết quả.

In [ ]:
def read_lineage(path, frames):
    mapping = pd.read_csv(path)
    columns = ["aug_row", "base_row"]
    if not set(columns).issubset(mapping):
        raise ValueError("Lineage cần hai cột aug_row và base_row.")
    numbers = mapping[columns].apply(pd.to_numeric, errors="coerce")
    for column, name in zip(columns, ("Augmented", "Base")):
        valid = numbers[column].isin(frames[name]["row"])
        if not valid.all():
            raise ValueError(f"Lineage có chỉ số {column} không hợp lệ.")
    numbers = numbers.astype(int).set_index("aug_row", verify_integrity=True)
    if set(numbers.index) != set(frames["Augmented"]["row"]):
        raise ValueError("Lineage phải ánh xạ đúng một lần cho mọi dòng Augmented.")
    parents = frames["Augmented"]["row"].map(numbers["base_row"])
    expected = parents.map(frames["Base"].set_index("row")["label"])
    if not expected.eq(frames["Augmented"]["label"]).all():
        raise ValueError("Lineage có dòng Augmented mang nhãn khác dòng Base cha.")
    return numbers["base_row"]

def unique_texts(frame):
    return frame.drop_duplicates("clean_key").reset_index(drop=True)

def partition_frames(development, frames, lineage_path):
    base = development["Base"]
    aug = development["Augmented"]
    validation = None
    if lineage_path is not None:
        parent_rows = read_lineage(lineage_path, frames)
        base_groups = frames["Base"].set_index("row")["text"].map(normalize_text).map(text_key)
        folds = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
        _, val_idx = next(folds.split(base, base["label"], groups=base["clean_key"]))
        validation = unique_texts(base.iloc[val_idx])
        val_keys = set(validation["clean_key"])
        val_raw_keys = set(validation["raw_key"])
        aug = aug.loc[aug["row"].map(parent_rows).isin(base["row"])]
        base = base.loc[~base["clean_key"].isin(val_keys)]
        parent_keys = aug["row"].map(parent_rows).map(base_groups)
        validation_descendants = aug.loc[parent_keys.isin(val_keys)]
        held_out_keys = val_keys | set(validation_descendants["clean_key"])
        held_out_raw = val_raw_keys | set(validation_descendants["raw_key"])
        aug = aug.loc[~(parent_keys.isin(val_keys) | aug["clean_key"].isin(held_out_keys)
                        | aug["raw_key"].isin(held_out_raw))]
    return {"Base": unique_texts(base), "Augmented": unique_texts(aug)}, validation

lineage_path = Path(AUGMENTATION_LINEAGE) if AUGMENTATION_LINEAGE is not None else None
if lineage_path is None and (data_root / "augmentation_lineage.csv").is_file():
    lineage_path = data_root / "augmentation_lineage.csv"
training, validation = partition_frames(cleaned, raw, lineage_path)
test = cleaned["Test"].reset_index(drop=True)

partitions = {**training, "Test": test}
if validation is not None:
    partitions["Validation"] = validation
for name, frame in partitions.items():
    if set(frame["label"]) != set(LABELS):
        raise ValueError(f"{name} cần đủ cả ba lớp sau kiểm toán và chia tập.")
for frame in training.values():
    for holdout in (test,) if validation is None else (test, validation):
        for key in ("raw_key", "clean_key"):
            if set(frame[key]) & set(holdout[key]):
                raise ValueError(f"Phát hiện giao nhau giữa train và holdout theo {key}.")

print("Chế độ:", "validation theo nhóm nguồn" if validation is not None else "cấu hình cố định, thiếu lineage")
audit["fit_or_test_rows"] = pd.Series({name: len(frame) for name, frame in {"Test": test, **training}.items()})
display(pd.DataFrame({name: frame["label"].value_counts() for name, frame in partitions.items()})
        .reindex(LABELS).fillna(0).astype(int))

### Phân bố nhãn và audit dữ liệu

Đọc số mẫu để biết tập nào mất nhiều dòng, rồi đối chiếu cột tỷ trọng để đánh giá mất cân bằng. Chênh lệch số dòng gốc và fit có thể gồm mẫu validation. Biểu đồ audit so sánh Base/Augmented trên cùng danh sách kiểm tra. Ví dụ một dòng vừa trùng nội bộ vừa giao test nên các cột không cộng thành tổng bị loại. Bảng lưu riêng số dòng trùng hoặc mâu thuẫn trong test phục vụ mô tả.

In [ ]:
stages = {
    "Base gốc": raw["Base"], "Base fit": training["Base"],
    "Augmented gốc": raw["Augmented"], "Augmented fit": training["Augmented"],
}
if validation is not None:
    stages["Validation"] = validation
label_counts = pd.DataFrame({
    stage: frame["label"].value_counts().reindex(LABELS, fill_value=0)
    for stage, frame in stages.items()
}).T.rename_axis("stage")
label_counts.to_csv(eda_dir / "label_counts.csv")
display(label_counts)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
label_counts.plot.bar(ax=axes[0], rot=20)
axes[0].set(ylabel="Số mẫu", xlabel="", title="Trước audit và sau lọc/chia tập")
label_counts.div(label_counts.sum(axis=1), axis=0).mul(100).plot.bar(stacked=True, ax=axes[1], rot=20)
axes[1].set(ylabel="Tỷ trọng (%)", xlabel="", title="Tỷ trọng nhãn")
save_text_figure(fig, "eda-labels.png")

audit_fields = {
    "raw_blank": "Rỗng gốc",
    "empty_after_cleaning": "Rỗng sau chuẩn hóa",
    "raw_duplicate_extras": "Trùng thêm (gốc)",
    "normalized_duplicate_extras": "Trùng thêm (chuẩn hóa)",
    "conflicting_dev_labels": "Mâu thuẫn nhãn (dòng dev)",
    "raw_overlap_with_test": "Giao test (gốc)",
    "normalized_overlap_with_test": "Giao test (chuẩn hóa)",
}
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True, constrained_layout=True)
max_audit_count = audit.loc[["Base", "Augmented"], list(audit_fields)].max().max()
for ax, name in zip(axes, ("Base", "Augmented")):
    values = audit.loc[name, list(audit_fields)].rename(index=audit_fields).astype(int)
    bars = ax.barh(list(values.index), values.to_numpy(), color="#3d749e")
    ax.bar_label(bars, padding=3)
    ax.set(xlabel="Số dòng · các nhóm có thể chồng lấn", title=name, xlim=(0, max(1, max_audit_count) * 1.18))
audit.to_csv(eda_dir / "data_audit.csv")
save_text_figure(fig, "eda-cleaning-audit.png")

## 5. Tách từ và đo chiều dài đầu vào

TF-IDF và Word2Vec dùng PyVi. PhoBERT-base-v2 dùng RDRSegmenter theo [hướng dẫn của VinAI](https://github.com/VinAIResearch/PhoBERT#notes) trước BPE. Thống kê p95, p99 và tỷ lệ vượt ngưỡng dùng đơn vị thực của từng nhánh, bao gồm token đặc biệt của PhoBERT. Các ngưỡng được cấu hình từ đầu để so sánh nhất quán. Thống kê BPE chạy cùng nhánh PhoBERT để Java chỉ khởi động khi cần.

In [ ]:
from functools import lru_cache
from pyvi import ViTokenizer

segment_words = lru_cache(maxsize=None)(ViTokenizer.tokenize)
for name, frame in partitions.items():
    if name == "Test" and not RUN_TRAINING:
        continue
    frame["words"] = frame["text_clean"].map(segment_words)

def length_summary(field, count_tokens, limit):
    rows = []
    for name, frame in partitions.items():
        if name == "Test":
            continue
        sizes = frame[field].map(count_tokens)
        rows.append({
            "split": name, "limit": limit, "median": sizes.median(),
            "p95": sizes.quantile(0.95), "p99": sizes.quantile(0.99),
            "over_limit": int(sizes.gt(limit).sum()),
            "over_limit_pct": 100 * sizes.gt(limit).mean(),
        })
    return pd.DataFrame(rows).set_index("split")

word_lengths = length_summary("words", lambda text: len(text.split()), W2V_MAX_LENGTH)
display(word_lengths.round(2))

### Từ vựng trước/sau chuẩn hóa và tương đồng lớp

Cả hai hình dùng Base fit. Ở hình từ vựng, phần trên dùng đúng cùng các văn bản trước/sau chuẩn hóa, đo phần trăm nhận xét có từ hoặc cụm từ. Phần dưới dùng vocabulary PyVi + TF-IDF để hiển thị bigram phổ biến. So sánh các dạng phủ định trước và sau khi gộp giúp giải thích vì sao chuẩn hóa teencode và giữ từ phủ định. Bigram bổ sung ngữ cảnh mà unigram có thể bỏ qua.

Heatmap đo cosine giữa vector TF-IDF trung bình của mỗi lớp. Ô ngoài đường chéo cao chỉ gợi ý hai lớp chia sẻ từ vựng. Đối chiếu giả thuyết này với confusion matrix khi có dự đoán thật. Các số liệu từ vựng được fit trên Base fit và chỉ dùng để mô tả.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

base_fit = training["Base"]
signals = ("không", "ko", "hok", "khg", "sp", "sản phẩm", "đc", "được")
signal_rows = []
for label, frame in base_fit.groupby("label"):
    for stage, field in (("Gốc", "text"), ("Chuẩn hóa", "text_clean")):
        for term in signals:
            pattern = r"(?<!\w)" + re.escape(term) + r"(?!\w)"
            share = frame[field].str.casefold().str.contains(pattern, regex=True).mean()
            signal_rows.append((label, stage, term, 100 * share))
signal_rates = pd.DataFrame(signal_rows, columns=["label", "stage", "term", "document_pct"])
signal_rates.to_csv(eda_dir / "lexical_signals.csv", index=False)

def make_tfidf_vectorizer():
    return TfidfVectorizer(
        max_features=5000, ngram_range=(1, 2), sublinear_tf=True, lowercase=False,
        token_pattern=r"(?u)\b\w+\b", dtype=np.float32,
    )

def plot_class_vocabulary(frame):
    vectorizer = make_tfidf_vectorizer()
    matrix = vectorizer.fit_transform(frame["words"])
    terms = vectorizer.get_feature_names_out()
    bigram_cols = np.flatnonzero(np.char.find(terms.astype(str), " ") >= 0)
    bigram_rows = []
    centroids = []
    for label in LABELS:
        sparse_rows = matrix[frame["label"].to_numpy() == label]
        centroids.append(np.asarray(sparse_rows.mean(axis=0)).ravel())
        rates = np.asarray(sparse_rows[:, bigram_cols].getnnz(axis=0)).ravel() / sparse_rows.shape[0]
        shown = sorted(np.flatnonzero(rates > 0), key=lambda index: (-rates[index], terms[bigram_cols[index]]))[:5]
        bigram_rows.extend((label, terms[bigram_cols[i]], 100 * rates[i]) for i in shown)

    bigrams = pd.DataFrame(bigram_rows, columns=["label", "bigram", "document_pct"])
    bigrams.to_csv(eda_dir / "top_bigrams.csv", index=False)
    fig, axes = plt.subplots(2, 3, figsize=(15, 7.5), constrained_layout=True)
    for col, label in enumerate(LABELS):
        values = signal_rates[signal_rates.label == label].pivot(index="term", columns="stage", values="document_pct")
        values.reindex(signals).plot.barh(ax=axes[0, col])
        axes[0, col].set(title=f"Nhãn {label} · Cùng các dòng Base fit", xlabel="% văn bản có từ/cụm từ", ylabel="")
        top = bigrams[bigrams.label == label].iloc[::-1]
        ax = axes[1, col]
        if top.empty:
            ax.text(0.5, 0.5, "Không có bigram trong vocabulary", transform=ax.transAxes, ha="center")
        else:
            ax.barh(top.bigram, top.document_pct, color="#256b62")
        ax.set(xlabel="% văn bản có bigram", title=f"Nhãn {label} · Bigram phổ biến")
    save_text_figure(fig, "eda-lexical.png")

    similarity = cosine_similarity(np.stack(centroids))
    pd.DataFrame(similarity, index=LABELS, columns=LABELS).to_csv(eda_dir / "class_cosine.csv")
    fig, ax = plt.subplots(figsize=(5.5, 4.5), constrained_layout=True)
    image = ax.imshow(similarity, vmin=0, vmax=1, cmap="Blues")
    ax.set(xticks=range(3), yticks=range(3), xticklabels=LABELS, yticklabels=LABELS,
           xlabel="Nhãn", ylabel="Nhãn", title="Base fit · Cosine giữa centroid TF-IDF")
    for row, col in np.ndindex(similarity.shape):
        value = similarity[row, col]
        ax.text(col, row, f"{value:.2f}", ha="center", va="center", color="white" if value > .6 else "black")
    fig.colorbar(image, ax=ax)
    save_text_figure(fig, "eda-class-similarity.png")
    pairs = [(similarity[i, j], LABELS[i], LABELS[j]) for i in range(3) for j in range(i + 1, 3)]
    score, left, right = max(pairs)
    print(f"Cặp ngoài đường chéo có cosine lớn nhất: {left} / {right} ({score:.3f}).")

plot_class_vocabulary(base_fit)

## 6. So sánh ba pipeline

Mỗi pipeline nhận hai tập fit Base và Augmented, với đầu vào test theo cùng thứ tự sample ID. Báo cáo accuracy, macro-F1, kết quả từng nhãn và ma trận nhầm lẫn. Bước dự đoán chỉ nhận văn bản đã biến đổi. Hàm đánh giá đối chiếu đầu ra với nhãn test sau khi dự đoán, và các chỉ số test dành cho báo cáo cuối.

LinearSVC fit ma trận đặc trưng theo tập. GRU và PhoBERT chạy mini-batch qua DataLoader hoặc Trainer. Báo cáo accuracy và macro-F1 cùng confusion matrix để xem lỗi theo từng nhãn. Khi có validation đã ghép nhóm, các nhánh neural dùng nó để chọn checkpoint; khi thiếu lineage, số epoch lấy từ cấu hình và được ghi trong quy trình.

In [ ]:
results = []
predictions = []

def sentiment_metrics(truth, predicted):
    truth, predicted = np.asarray(truth), np.asarray(predicted)
    report = classification_report(truth, predicted, labels=LABELS, output_dict=True, zero_division=0)
    summary = {"accuracy": float(accuracy_score(truth, predicted)),
               "balanced_accuracy": float(balanced_accuracy_score(truth, predicted)),
               "mcc": float(matthews_corrcoef(truth, predicted))}
    for averaging in ("macro", "weighted"):
        for source, target in (("precision", "precision"), ("recall", "recall"), ("f1-score", "f1")):
            summary[f"{averaging}_{target}"] = float(report[f"{averaging} avg"][source])

    # Tỷ lệ đổi cực trong các mẫu thật có nhãn tiêu cực hoặc tích cực.
    polarized = truth != 0
    summary["polarity_reversal_rate"] = (
        float(np.mean(predicted[polarized] == -truth[polarized])) if polarized.any() else None
    )
    return summary, report


def per_class_rows(results):
    rows = []
    for result in results:
        report = result["classification_report"]
        for label, class_name in zip(LABELS, ("negative", "neutral", "positive")):
            values = report[str(label)]
            rows.append({
                "pipeline": result["pipeline"], "dataset": result["dataset"], "label": label,
                "class_name": class_name, "precision": values["precision"], "recall": values["recall"],
                "f1": values["f1-score"], "support": int(values["support"]),
            })
    return rows


def record_result(pipeline, dataset, predicted, started):
    predicted = np.asarray(predicted)
    if predicted.shape != (len(test),) or not np.isin(predicted, LABELS).all():
        raise ValueError(f"Dự đoán không hợp lệ cho {pipeline}/{dataset}.")
    elapsed = time.perf_counter() - started
    truth = test["label"].to_numpy()
    matrix = confusion_matrix(truth, predicted, labels=LABELS)
    summary, report = sentiment_metrics(truth, predicted)
    if matrix.sum() != len(test) or not np.isclose(matrix.trace() / matrix.sum(), summary["accuracy"]):
        raise ValueError("Ma trận nhầm lẫn không nhất quán với accuracy.")
    result = {
        "pipeline": pipeline,
        "dataset": dataset,
        **summary,
        "elapsed_seconds": elapsed,
        "confusion_matrix": matrix.tolist(),
        "classification_report": report,
    }
    prediction_rows = pd.DataFrame({
        "pipeline": pipeline, "dataset": dataset, "sample_id": test["sample_id"],
        "true_label": truth, "predicted_label": predicted,
    })
    for index, previous in enumerate(results):
        if (previous["pipeline"], previous["dataset"]) == (pipeline, dataset):
            results[index] = result
            predictions[index] = prediction_rows
            return
    results.append(result)
    predictions.append(prediction_rows)

### 6.1. TF-IDF → TruncatedSVD → LinearSVC

Dùng unigram và bigram để giữ cụm mang sắc thái, chẳng hạn tổ hợp có “không”. Fit TF-IDF và TruncatedSVD riêng trên từng tập train. Ma trận TF-IDF giữ dạng thưa và dùng float32 cho đến bước giảm chiều. Nhật ký hiển thị bước đang chạy và kích thước ma trận. Số thành phần tối đa được đặt ở `TFIDF_SVD_COMPONENTS`.

In [ ]:
from threadpoolctl import threadpool_limits

def predict_tfidf_svc(name, frame, test_words):
    print(f"[{name}] Fit TF-IDF trên {len(frame):,} văn bản", flush=True)
    vectorizer = make_tfidf_vectorizer()
    try:
        fit_vectors = vectorizer.fit_transform(frame["words"])
    except ValueError as error:
        raise ValueError(f"[{name}] Không thể fit TF-IDF: {error}") from error

    rows, features = fit_vectors.shape
    components = min(TFIDF_SVD_COMPONENTS, rows - 1, features - 1)
    if components < 1:
        raise ValueError(
            f"[{name}] Không thể giảm chiều TF-IDF từ ma trận {rows} × {features} "
            f"với TFIDF_SVD_COMPONENTS={TFIDF_SVD_COMPONENTS}."
        )

    print(f"[{name}] Fit SVD: {rows:,} dòng × {features:,} đặc trưng → {components} chiều", flush=True)
    reducer = TruncatedSVD(n_components=components, random_state=SEED)
    with threadpool_limits(limits=2, user_api="blas"):
        reduced_train = reducer.fit_transform(fit_vectors)
        del fit_vectors

        print(f"[{name}] Fit LinearSVC", flush=True)
        model = LinearSVC(random_state=SEED, max_iter=5000)
        model.fit(reduced_train, frame["label"])
        del reduced_train

        print(f"[{name}] Dự đoán {len(test_words):,} mẫu test", flush=True)
        return model.predict(reducer.transform(vectorizer.transform(test_words)))


if RUN_TRAINING:
    for name, frame in training.items():
        started = time.perf_counter()
        predicted = predict_tfidf_svc(name, frame, test["words"])
        record_result("TF-IDF → TruncatedSVD → LinearSVC", name, predicted, started)
        print(f"[{name}] Hoàn tất sau {time.perf_counter() - started:.1f} giây", flush=True)

### 6.2. Word2Vec CBOW + GRU

Vocabulary học riêng trên từng tập fit. Mỗi từ PyVi giữ một vị trí trong chuỗi, dùng vector CBOW tiền huấn luyện khi có. Từ chưa có vector và token UNK được khởi tạo theo seed. GRU nhận trực tiếp chuỗi vector từ theo thứ tự gốc, với hidden size 64. Chiều đầu vào lấy từ mô hình CBOW, nên pipeline không cần bước PCA.

DataLoader tạo mini-batch 64 câu, cắt ở 256 từ và đệm bên phải. GRU chỉ đọc các vị trí hợp lệ rồi dùng trạng thái ẩn cuối để phân loại ba nhãn. Huấn luyện bằng Adam và CrossEntropyLoss tối đa bốn epoch. Khi có validation theo nhóm, nạp lại trạng thái có macro-F1 validation cao nhất. Với cấu hình cố định, dùng trạng thái sau epoch cuối.

In [ ]:
if RUN_TRAINING:
    import copy

    import torch
    from torch import nn
    from torch.nn.utils.rnn import PackedSequence, pack_padded_sequence
    from torch.utils.data import DataLoader, TensorDataset

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    vector_path = Path(WORD2VEC_FILE) if WORD2VEC_FILE is not None else data_root / "vi-model-CBOW.bin"
    if not vector_path.is_file():
        raise FileNotFoundError(f"Thiếu CBOW tại {vector_path}. Đặt WORD2VEC_FILE tới vi-model-CBOW.bin.")

    def make_vocabulary(frame):
        counts = Counter(token for text in frame["words"] for token in text.split())
        return {word: index for index, (word, _) in enumerate(counts.most_common(9998), start=2)}

    def embedding_weights(vocabulary, word_vectors):
        rng = np.random.default_rng(SEED)
        weights = rng.normal(0, 0.05, size=(len(vocabulary) + 2, word_vectors.vector_size)).astype(np.float32)
        weights[0] = 0
        for word, index in vocabulary.items():
            if word in word_vectors:
                weights[index] = word_vectors[word]
        return torch.from_numpy(weights)

    def text_tensors(frame, vocabulary):
        data = np.zeros((len(frame), W2V_MAX_LENGTH), dtype=np.int64)
        lengths = []
        for row, text in enumerate(frame["words"]):
            sequence = [vocabulary.get(token, 1) for token in text.split()[:W2V_MAX_LENGTH]] or [1]
            lengths.append(len(sequence))
            data[row, :len(sequence)] = sequence
        return torch.from_numpy(data), torch.tensor(lengths, dtype=torch.long)

    class CBOWGRU(nn.Module):
        def __init__(self, embeddings):
            super().__init__()
            self.embedding = nn.Embedding.from_pretrained(embeddings, freeze=False, padding_idx=0)
            self.gru = nn.GRU(embeddings.size(1), GRU_HIDDEN_SIZE, batch_first=True)
            self.classifier = nn.Sequential(nn.Dropout(0.5), nn.Linear(GRU_HIDDEN_SIZE, len(LABELS)))

        def forward(self, inputs, lengths):
            packed = pack_padded_sequence(inputs, lengths.cpu(), batch_first=True, enforce_sorted=False)
            embedded = PackedSequence(
                self.embedding(packed.data), packed.batch_sizes, packed.sorted_indices, packed.unsorted_indices
            )
            _, hidden = self.gru(embedded)
            return self.classifier(hidden[-1])

In [ ]:
if RUN_TRAINING:
    from gensim.models import KeyedVectors

    @torch.inference_mode()
    def predict_gru(model, loader):
        model.eval()
        return np.concatenate([
            model(tokens.to(DEVICE), lengths).argmax(dim=-1).cpu().numpy() - 1
            for tokens, lengths in loader
        ])

    def train_gru(frame, vocabulary, word_vectors):
        torch.manual_seed(SEED)
        model = CBOWGRU(embedding_weights(vocabulary, word_vectors)).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
        loss_fn = nn.CrossEntropyLoss()
        batches = TensorDataset(*text_tensors(frame, vocabulary), torch.tensor(frame["label"].to_numpy() + 1))
        loader = DataLoader(batches, batch_size=GRU_BATCH_SIZE, shuffle=True, generator=torch.Generator().manual_seed(SEED))
        if validation is not None:
            val_loader = DataLoader(TensorDataset(*text_tensors(validation, vocabulary)), batch_size=GRU_BATCH_SIZE)
        best_score, best_state = -1.0, None
        for _ in range(GRU_EPOCHS):
            model.train()
            for tokens, lengths, labels in loader:
                optimizer.zero_grad(set_to_none=True)
                loss = loss_fn(model(tokens.to(DEVICE), lengths), labels.to(DEVICE))
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            if validation is not None:
                score = f1_score(validation["label"], predict_gru(model, val_loader), labels=LABELS,
                                 average="macro", zero_division=0)
                if score > best_score:
                    best_score = score
                    best_state = copy.deepcopy(model.state_dict())
        if best_state is not None:
            model.load_state_dict(best_state)
        return model

    word_vectors = KeyedVectors.load_word2vec_format(str(vector_path), binary=True)
    for name, frame in training.items():
        started = time.perf_counter()
        print(f"[{name}] Huấn luyện CBOW + GRU", flush=True)
        vocabulary = make_vocabulary(frame)
        model = train_gru(frame, vocabulary, word_vectors)
        test_loader = DataLoader(TensorDataset(*text_tensors(test, vocabulary)), batch_size=GRU_BATCH_SIZE)
        record_result("Word2Vec CBOW + GRU", name, predict_gru(model, test_loader), started)
        del model

    del word_vectors
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

### 6.3. PhoBERT-base-v2

Cắt chuỗi BPE tối đa 256 token và đệm theo độ dài lớn nhất trong mỗi batch. Cả hai phiên bản bắt đầu từ cùng checkpoint pretrained. Với validation theo nhóm, Trainer giữ checkpoint có macro-F1 validation cao nhất. Khi tập validation chưa có lineage, dùng ba epoch cố định. Dataset truyền vào bước dự đoán test chỉ có đầu vào tokenizer.

In [ ]:
import py_vncorenlp
from transformers import AutoTokenizer

java_executable = shutil.which("java")
if java_executable is None:
    raise RuntimeError("Cần Java để chạy RDRSegmenter của VnCoreNLP.")
os.environ.setdefault("JAVA_HOME", str(Path(java_executable).resolve().parent.parent))
segmenter_dir = Path("vncorenlp").resolve()
segmenter_dir.mkdir(exist_ok=True)
notebook_dir = Path.cwd()
try:
    if not (segmenter_dir / "VnCoreNLP-1.2.jar").is_file():
        py_vncorenlp.download_model(save_dir=str(segmenter_dir))
    segmenter = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=str(segmenter_dir))
finally:
    os.chdir(notebook_dir)

@lru_cache(maxsize=None)
def phobert_words(text):
    return " ".join(segmenter.word_segment(text))

for name, frame in partitions.items():
    if name == "Test" and not RUN_TRAINING:
        continue
    frame["phobert_words"] = frame["text_clean"].map(phobert_words)

phobert_tokenizer = AutoTokenizer.from_pretrained(PHOBERT_MODEL, use_fast=False)
bpe_lengths = length_summary(
    "phobert_words",
    lambda text: len(phobert_tokenizer.tokenize(text)) + phobert_tokenizer.num_special_tokens_to_add(),
    PHOBERT_MAX_LENGTH,
)
display(bpe_lengths.round(2))

### Kiểm tra ngưỡng đầu vào theo tokenizer

Mỗi ô hiển thị trung vị, p95, p99 và tỷ lệ vượt ngưỡng trong các tập phát triển. Các giá trị của PyVi và PhoBERT có đơn vị khác nhau. PhoBERT tính cả token đặc biệt. Đường ngang đánh dấu ngưỡng cắt thực sự để lượng hóa bao nhiêu mẫu có thể mất phần cuối. So sánh Base và Augmented trước khi cân nhắc đổi ngưỡng bằng validation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
for ax, table, title in zip(axes, (word_lengths, bpe_lengths), ("Từ sau PyVi", "Token BPE (gồm token đặc biệt)")):
    table[["median", "p95", "p99"]].plot.bar(ax=ax, rot=15)
    limit = int(table["limit"].iloc[0])
    ax.axhline(limit, color="#923d29", linestyle="--", label=f"Ngưỡng {limit}")
    ax.set(title=title, xlabel="", ylabel="Độ dài")
    ax.set_xticklabels([f"{name}\n{row.over_limit_pct:.2f}% vượt ngưỡng" for name, row in table.iterrows()])
    ax.legend()
word_lengths.to_csv(eda_dir / "word_lengths.csv")
bpe_lengths.to_csv(eda_dir / "bpe_lengths.csv")
save_text_figure(fig, "eda-tokenizers.png")

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

eda_sources = {**paths, **({"Lineage": lineage_path} if lineage_path is not None else {})}
(eda_dir / "eda_source.json").write_text(json.dumps({
    "dataset_handle": DATASET_HANDLE, "seed": SEED, "phobert_tokenizer": PHOBERT_MODEL,
    "inputs": {name: {"file": path.name, "sha256": sha256_file(path)} for name, path in eda_sources.items()},
    "fit_mode": "grouped_validation" if validation is not None else "fixed_configuration",
    "figures": sorted(path.name for path in eda_dir.glob("*.png")),
}, indent=2), encoding="utf-8")
print("Đã xuất EDA tại", eda_dir.resolve())

In [ ]:
if RUN_TRAINING:
    from datasets import Dataset
    from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments, set_seed

    def transformer_dataset(frame, include_labels=True):
        encoded = dict(phobert_tokenizer(
            frame["phobert_words"].tolist(), truncation=True, max_length=PHOBERT_MAX_LENGTH,
        ))
        if include_labels:
            encoded["labels"] = (frame["label"] + 1).tolist()
        return Dataset.from_dict(encoded)

    def compute_metrics(evaluation):
        predictions, labels = evaluation
        return {"macro_f1": f1_score(labels, predictions.argmax(axis=-1), labels=[0, 1, 2],
                                    average="macro", zero_division=0)}

    transformer_test = transformer_dataset(test, include_labels=False)
    transformer_validation = transformer_dataset(validation) if validation is not None else None

    for name, frame in training.items():
        started = time.perf_counter()
        set_seed(SEED)
        model = AutoModelForSequenceClassification.from_pretrained(PHOBERT_MODEL, num_labels=len(LABELS))
        use_validation = transformer_validation is not None
        arguments = TrainingArguments(
            output_dir=str(OUTPUT_PARENT / "checkpoints" / name.lower()),
            learning_rate=2e-5, weight_decay=0.01, warmup_ratio=0.1, num_train_epochs=3,
            per_device_train_batch_size=8, gradient_accumulation_steps=4,
            per_device_eval_batch_size=16, seed=SEED, data_seed=SEED, report_to="none",
            eval_strategy="epoch" if use_validation else "no",
            save_strategy="epoch" if use_validation else "no",
            load_best_model_at_end=use_validation, metric_for_best_model="macro_f1" if use_validation else None,
            save_total_limit=1, fp16=DEVICE.type == "cuda",
        )
        trainer = Trainer(
            model=model, args=arguments, train_dataset=transformer_dataset(frame),
            eval_dataset=transformer_validation,
            data_collator=DataCollatorWithPadding(phobert_tokenizer), compute_metrics=compute_metrics,
        )
        trainer.train()
        predicted = trainer.predict(transformer_test).predictions.argmax(axis=-1) - 1
        record_result("PhoBERT-base-v2", name, predicted, started)
        del trainer, model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

## 7. Xuất bằng chứng và đọc kết quả

JSON và CSV tóm tắt, ma trận nhầm lẫn, biểu đồ so sánh và danh sách dự đoán theo ID cùng lấy từ một lượt chạy. Thư mục kết quả chứa hash các file đầu vào và báo cáo cắt chuỗi. Chênh lệch Macro F1 được tính theo điểm phần trăm. Mỗi cấu hình chỉ chạy một seed, và kiểm tra giao nhau chỉ phát hiện bản trùng chính xác theo khóa đã chuẩn hóa.

Bảng tổng hợp báo accuracy, balanced accuracy, precision/recall/F1 theo macro và weighted, cùng MCC (Matthews correlation coefficient). Bảng theo nhãn báo precision, recall, F1 và support cho tiêu cực, trung lập và tích cực. Tỷ lệ đổi cực là phần mẫu tiêu cực hoặc tích cực bị dự đoán sang cực đối diện, chia cho tất cả mẫu thật thuộc hai cực. Kết quả tính theo tỷ lệ 0–1, riêng MCC nằm trong khoảng −1 đến 1.

Các bảng đã lưu được tính lại từ số đếm của sáu ma trận nhầm lẫn của bản chạy Colab. Thời gian và hình giữ theo output gốc. Chạy lại từ đầu để xuất đầy đủ file của một lượt chạy mới.

In [ ]:
if RUN_TRAINING:
    import tempfile

    expected_runs = {(pipeline, name) for pipeline in {
        "TF-IDF → TruncatedSVD → LinearSVC",
        "Word2Vec CBOW + GRU",
        "PhoBERT-base-v2",
    } for name in training}
    runs = {(r["pipeline"], r["dataset"]): r for r in results}
    if len(results) != len(expected_runs) or set(runs) != expected_runs:
        raise RuntimeError("Cần đủ đúng sáu lượt dự đoán trước khi xuất kết quả. Chạy lại notebook từ đầu.")

    OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)
    output_dir = Path(tempfile.mkdtemp(prefix="run-", dir=OUTPUT_PARENT))

    input_paths = {**paths, "Word2Vec": vector_path}
    if lineage_path is not None:
        input_paths["Lineage"] = lineage_path
    metadata = {
        "seed": SEED,
        "dataset_handle": DATASET_HANDLE,
        "inputs": {name: {"file": path.name, "sha256": sha256_file(path)} for name, path in input_paths.items()},
        "validation": "grouped" if validation is not None else "fixed_configuration",
        "label_order": LABELS,
        "phobert_model": PHOBERT_MODEL,
        "tfidf_svd_max_components": TFIDF_SVD_COMPONENTS,
        "cbow_gru": {
            "hidden_size": GRU_HIDDEN_SIZE, "batch_size": GRU_BATCH_SIZE,
            "epochs": GRU_EPOCHS, "max_length": W2V_MAX_LENGTH,
        },
        "sample_counts": {name: len(frame) for name, frame in partitions.items()},
    }
    (output_dir / "pipeline_comparison_results.json").write_text(
        json.dumps({"run": metadata, "results": results}, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    summary = pd.DataFrame(results).drop(columns=["confusion_matrix", "classification_report"])
    summary.to_csv(output_dir / "pipeline_comparison_results.csv", index=False)
    per_class = pd.DataFrame(per_class_rows(results))
    per_class.to_csv(output_dir / "per_class_metrics.csv", index=False)
    pd.concat(predictions, ignore_index=True).to_csv(output_dir / "predictions.csv", index=False)
    pd.concat([
        frame.assign(split=name)[["split", "sample_id", "label"]]
        for name, frame in partitions.items()
    ]).to_csv(output_dir / "split_manifest.csv", index=False)
    audit.to_csv(output_dir / "data_audit.csv")
    word_lengths.to_csv(output_dir / "word_lengths.csv")
    bpe_lengths.to_csv(output_dir / "bpe_lengths.csv")
    display(summary.round(4))
    display(per_class.round(4))
    comparison = summary.pivot(index="pipeline", columns="dataset", values="macro_f1")[["Base", "Augmented"]]
    display(comparison.assign(delta_pp=100 * (comparison["Augmented"] - comparison["Base"])).round(4))

    ax = comparison.plot.barh(figsize=(10, 4), xlim=(0, 1), xlabel="Macro F1")
    ax.figure.tight_layout()
    ax.figure.savefig(output_dir / "model-comparison.png", dpi=150)
    plt.show()

    fig, axes = plt.subplots(
        len(comparison.index), len(comparison.columns),
        figsize=(10, 3.5 * len(comparison.index)), squeeze=False, constrained_layout=True,
    )
    ordered_runs = [runs[(pipeline, dataset)] for pipeline in comparison.index for dataset in comparison.columns]
    for ax, result in zip(axes.flat, ordered_runs):
        matrix = np.asarray(result["confusion_matrix"])
        ax.imshow(matrix, cmap="Blues", vmin=0, vmax=max(test["label"].value_counts()))
        ax.set(xticks=range(3), yticks=range(3), xticklabels=LABELS, yticklabels=LABELS,
               xlabel="Dự đoán", ylabel="Nhãn thật",
               title=f'{result["pipeline"]}\n{result["dataset"]}')
        for row in range(3):
            for column in range(3):
                ax.text(column, row, matrix[row, column], ha="center", va="center",
                        color="white" if matrix[row, column] > matrix.max() * 0.55 else "black")
    fig.savefig(output_dir / "confusion-matrices.png", dpi=150)
    plt.show()
    print(f"Kết quả đã lưu tại {output_dir.resolve()}")